In [ ]:
!pip install -q langchain langchain-community chromadb langchain-huggingface


## Use this for llama3 ##

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_community.embeddings import OllamaEmbeddings

# embeddings
embeddings_llama = OllamaEmbeddings(model="llama3")
# llm
llm = Ollama(model="llama3", temperature=0.2)

In [ ]:
knowledge_base = [
    "RAG stands for Retrieval-Augmented Generation. It combines a retriever with a generator.",
    "The retrieval step searches a vector database to find text chunks relevant to a user's query.",
    "Embeddings turn text into numerical vectors so that semantic similarity can be measured mathematically.",
    "Cosine similarity measures the angle between two vectors — closer to 1 means more similar meaning.",
    "The augmentation step combines the retrieved chunks with the original query into a single prompt.",
    "The generation step passes the augmented prompt to a large language model to produce the final answer.",
    "RAG helps reduce hallucination because the model answers using real retrieved facts instead of memory alone.",
    "The capital of France is Paris, a city known for the Eiffel Tower and the Louvre museum.",
    "Eiffel tower is one of the most iconic landmarks in Paris, France.",
    "Photosynthesis is the process plants use to convert sunlight into chemical energy.",
]

vectorstore = Chroma.from_texts(
            texts = knowledge_base, 
            embedding=embeddings_llama,
            collection_name= "rag_demo_llama")


## Use this for tinyllama ##

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_huggingface import HuggingFaceEmbeddings


# LLM
llm_tinyllama = Ollama(model="tinyllama", temperature=0.2)

# Embedding model
embedding_tinyllama = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


In [ ]:
knowledge_base = [
    "RAG stands for Retrieval-Augmented Generation. It combines a retriever with a generator.",
    "The retrieval step searches a vector database to find text chunks relevant to a user's query.",
    "Embeddings turn text into numerical vectors so that semantic similarity can be measured mathematically.",
    "Cosine similarity measures the angle between two vectors — closer to 1 means more similar meaning.",
    "The augmentation step combines the retrieved chunks with the original query into a single prompt.",
    "The generation step passes the augmented prompt to a large language model to produce the final answer.",
    "RAG helps reduce hallucination because the model answers using real retrieved facts instead of memory alone.",
    "The capital of France is Paris, a city known for the Eiffel Tower and the Louvre museum.",
    "Eiffel tower is one of the most iconic landmarks in Paris, France.",
    "Photosynthesis is the process plants use to convert sunlight into chemical energy.",
]


vectorstore_tinyllama = Chroma.from_texts(
    texts=knowledge_base, embedding=embedding_tinyllama, collection_name="rag_demo_tinyllama"

## Code from here can be used for both ##

In [ ]:
def retrieve(query, top_k=3):
    results = vectorstore_tinyllama.similarity_search_with_score(query, k=top_k)
    return [(doc.page_content, score) for doc, score in results]


query = "Explain about Eiffel Tower"
result = retrieve(query, top_k=1)

for i, (doc, score) in enumerate(result, 1):
    print(f"--- Retrieved Chunk {i} ---")
    print(f"Score: {score:.4f}")
    print(doc)

## Prompt Engineering ##

In [ ]:
query_1 = "Explain about Eiffel Tower"

In [ ]:
# Zero shot prompt template

def zero_shot(query,retrieved_docs):
    context = "/n".join([f"-{doc}" for doc,score in retrieved_docs])

    prompt = f"""
    Use the following context to answer the question. If the answer is not contained within the context, respond with "I don't know".
    
Context:
{context}

Question: {query}
Answer: 
"""
    return prompt

zero_shot_prompt = zero_shot(query_1,result)
print(zero_shot_prompt)
    
    

In [ ]:
#few_shot_templete

def few_shot(query,retrieved_docs):
    context = "/n".join([f"-{doc}" for doc,score in retrieved_docs])
    
    examples = """
    
Example 1:

Context: Everest is the highest mountain in the world.It is located in the Himalayas on the border between Nepal and China. It has an elevation of 8,848 meters (29,029 feet) above sea level.
Question: What is the highest mountain in the world?
Answer: Everest is the highest mountain in the world.


Example 2:

Context: The capital of France is Paris, a city known for the Eiffel Tower and the Louvre museum.
Question: What is the capital of France?
Answer: The capital of France is Paris.

"""

    prompt = f""" Answer the question based on the context and examples provided. If the answer is not contained within the context, respond with "I don't know".
    {examples}

Context:
{context}
Question: {query}
Answer:
   """
    return prompt

few_shot_prompt = few_shot(query_1,result)
print(few_shot_prompt)

In [ ]:
#chain of thought / adaptive learning prompt template

## Finetuning , Self hosted LLM train. 

def chain_of_thought(query,retrieved_docs):
    context = "/n".join([f"-{doc}" for doc,score in retrieved_docs])
    
    prompt = f"""
    Use the following context to answer the question. Think step by step and explain your reasoning in a "Reasoning:" section before providing the final answer.
    If the answer is not contained within the context, respond with "I don't know".

    Context: 
    {context}

Question: {query}

Reasoning:
    """
    return prompt

cot_prompt = chain_of_thought(query_1,result)
print(cot_prompt)

## Generation ##

In [ ]:
def generate_answer(prompt):
    return llm.invoke(prompt)



In [ ]:
zero_shot_answer = generate_answer(zero_shot_prompt)

In [ ]:
print(zero_shot_answer)

In [ ]:
few_shot_answer = generate_answer(few_shot_prompt)
print(few_shot_answer)

In [ ]:
cot_answer = generate_answer(cot_prompt)
print(cot_answer)